# SFT Distillation: Carnice-9B on Claude Reasoning Traces (Unsloth)

**Author:** Behrooz Azarkhalili

Fine-tunes **Carnice-9B** (Qwen3.5-9B Hermes-Agent tuned) on
[claude-reasoning-distillation](https://huggingface.co/datasets/ermiaazarkhalili/claude-reasoning-distillation)
using **Unsloth** for 2x faster training.

| Feature | Detail |
|---------|--------|
| Framework | Unsloth (FastLanguageModel) |
| Model | `kai-os/Carnice-9b` (Qwen3.5-9B base, Hermes-Agent tuned) |
| Size | 16.7 GB fp16 → ~5 GB 4-bit |
| Architecture | qwen3_5_text (standard LM) |
| Dataset | Claude Reasoning Distillation (~10K samples with thinking traces) |
| Method | LoRA (4-bit QLoRA) |
| GGUF Export | Built-in via `save_pretrained_gguf()` |

### Hardware
- **Minimum:** 12 GB VRAM (QLoRA, batch_size=1)
- **Recommended:** 40+ GB for batch_size > 1

### Why Carnice-9B?
Already tuned on Hermes Agent traces (tool use, terminal, browser) + reasoning data
(NuminaMath-CoT, Bespoke-Stratos). Further training on Claude reasoning should
sharpen rather than replace its agentic capabilities.

In [ ]:
# ============================================================================
# Install (uncomment for Colab / bare-metal)
# ============================================================================
# !pip install unsloth datasets tqdm

# SLURM (Fir cluster):
#   module load gcc arrow python/3.11.5
#   source /scratch/$USER/venvs/hf_unsloth/bin/activate

In [ ]:
from __future__ import annotations
import json, os, torch

SMOKE_TEST: bool = False  # Set False for full training

MODEL_NAME = "kai-os/Carnice-9b"
# MODEL_NAME = "kai-os/Carnice-9b"  # Alternative
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
HUB_MODEL_ID = "ermiaazarkhalili/Carnice-9B-SFT-Claude-Opus-Reasoning-Unsloth"
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
MAX_STEPS = 5 if SMOKE_TEST else 1000
BATCH_SIZE = 1
GRAD_ACCUM = 8

print(f"[OK] Model: {MODEL_NAME}")
print(f"[OK] SMOKE_TEST: {SMOKE_TEST}, max_steps: {MAX_STEPS}")
print(f"[OK] PyTorch: {torch.__version__}")
print(f"[OK] GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ============================================================================
# HF Authentication
# ============================================================================
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError: pass
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception: pass
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("[OK] Authenticated with HF Hub")
else:
    print("[WARN] No HF_TOKEN found")

In [ ]:
# ============================================================================
# Load Model with Unsloth
# ============================================================================
# Qwen3.5 returns a processor (VLM arch) — extract tokenizer for text-only use
from unsloth import FastLanguageModel

model, processor = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
tokenizer = processor.tokenizer
print(f"[OK] Model loaded: {MODEL_NAME}")

In [ ]:
# ============================================================================
# Apply LoRA
# ============================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none", random_state=3407,
)
print(f"[OK] LoRA applied: r={LORA_R}, alpha={LORA_ALPHA}")

In [ ]:
# ============================================================================
# Chat Template (Qwen3.5 uses built-in template — no get_chat_template needed)
# ============================================================================
print(f"[OK] Using built-in Qwen3.5 chat template")

In [ ]:
# ============================================================================
# Load & Process Claude Reasoning Distillation Dataset
# ============================================================================
from datasets import load_dataset

dataset = load_dataset("ermiaazarkhalili/claude-reasoning-distillation", "sft", split="train")
print(f"[OK] Loaded {len(dataset):,} samples")
if SMOKE_TEST:
    dataset = dataset.select(range(100))
    print(f"[SMOKE] Truncated to {len(dataset)} samples")

def format_distillation(sample):
    messages = []
    for msg in sample["messages"]:
        m = dict(msg)
        m.pop("thinking", None)  # Drop thinking field — model learns from content
        messages.append(m)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_distillation)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
print(f"[OK] Processed: {len(dataset):,} samples")
print(dataset[0]["text"][:200])

In [ ]:
# ============================================================================
# Training
# ============================================================================
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset, eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5, max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE, logging_steps=1,
        optim="adamw_8bit", weight_decay=0.001,
        lr_scheduler_type="linear", seed=3407,
        output_dir="outputs", report_to="none",
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
start_mem = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"[OK] GPU: {gpu_stats.name}, {max_memory} GB, reserved: {start_mem} GB")
print(f"[....] Training ({MAX_STEPS} steps) ...")

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n[OK] Training complete!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"  Peak VRAM: {used} GB ({round(used/max_memory*100, 1)}%)")

In [ ]:
# ============================================================================
# Inference Test
# ============================================================================
test_prompts = [
    "Check if the numbers 8 and 1233 are powers of two.",
    "What's the weather like in New York today?",
    "Calculate the average of: 10, 20, 30, 40, 50",
]

print("[....] Testing ...\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{'=' * 50}")
    print(f"Test {i}: {prompt}")
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        max_new_tokens=256, temperature=0.7, top_p=0.8, top_k=20, use_cache=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the generated part
    if prompt in response:
        response = response.split(prompt)[-1]
    print(response[:300])
    print()

In [ ]:
# ============================================================================
# Save & Export
# ============================================================================
model.save_pretrained("carnice_9b_distill_lora")
tokenizer.save_pretrained("carnice_9b_distill_lora")
print("[OK] LoRA adapter saved")

if not SMOKE_TEST:
    model.save_pretrained_merged("carnice_9b_distill_merged", tokenizer)
    print("[OK] Merged model saved")
    model.push_to_hub_merged(HUB_MODEL_ID, tokenizer, token=os.environ.get("HF_TOKEN", ""))
    print(f"[OK] Pushed to {HUB_MODEL_ID}")
    try:
        model.save_pretrained_gguf("carnice_9b_distill_gguf", tokenizer, quantization_method=["q4_k_m", "q5_k_m", "q8_0"])
        print("[OK] GGUF saved")
        model.push_to_hub_gguf(f"{HUB_MODEL_ID}-GGUF", tokenizer, quantization_method=["q4_k_m", "q5_k_m", "q8_0"], token=os.environ.get("HF_TOKEN", ""))
        print(f"[OK] GGUF pushed")
    except Exception as e:
        print(f"[WARN] GGUF export failed (non-fatal): {e}")
        print("[INFO] Training + merge + Hub push succeeded. GGUF can be done separately.")
else:
    print("[SMOKE] Skipping merge/GGUF/push")

print("\n[OK] Done!")